# Converting a trained model to tflite
https://www.tensorflow.org/lite/microcontrollers/build_convert#model_conversion

# Convert model to tflite

In [1]:
import tensorflow as tf
import numpy as np

In [2]:
training_spectrogram = np.load('training_spectrogram.npz')
validation_spectrogram = np.load('validation_spectrogram.npz')
test_spectrogram = np.load('test_spectrogram.npz')

X_train = training_spectrogram['X']
X_validate = validation_spectrogram['X']
X_test = test_spectrogram['X']

complete_train_X = np.concatenate((X_train, X_validate, X_test))
# complete_train_X = X_validate

In [3]:
import os


# .keras is a Keras model file/folder, not a SavedModel export directory.
#model_candidates = ["fully_trained.model.keras", "trained.model.keras"]
model_candidates = ["fully_trained.model.keras"]
model_path = next((p for p in model_candidates if os.path.exists(p)), None)
if model_path is None:
    raise FileNotFoundError("No .keras model found. Expected one of: fully_trained.model.keras, trained.model.keras")

keras_model = tf.keras.models.load_model(model_path)
keras_model.trainable = False

# Build a fixed-batch inference model (batch=1) to keep shapes static for TFLM.
fixed_input = tf.keras.Input(
    shape=keras_model.input_shape[1:],
    batch_size=1,
    dtype=tf.float32,
    name="input"
 )
fixed_output = keras_model(fixed_input, training=False)
inference_model = tf.keras.Model(inputs=fixed_input, outputs=fixed_output, name="inference_model")

converter2 = tf.lite.TFLiteConverter.from_keras_model(inference_model)
converter2.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_dataset_gen():
    # Use single examples to match fixed batch size 1.
    for i in range(0, len(complete_train_X), 100):
        sample = complete_train_X[i]
        if sample.ndim == len(inference_model.input_shape) - 1:
            sample = np.expand_dims(sample, axis=0)
        yield [sample.astype(np.float32)]

converter2.representative_dataset = representative_dataset_gen
converter2.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter2.inference_input_type = tf.int8
converter2.inference_output_type = tf.int8

# Keep the MLIR converter path explicit; it is the stable path for TF 2.21.
converter2.experimental_new_converter = True

tflite_quant_model = converter2.convert()
open("converted_model.tflite", "wb").write(tflite_quant_model)
print(f"Converted {model_path} -> converted_model.tflite ({len(tflite_quant_model)} bytes)")

# Quick sanity check of required ops in the generated model.
interpreter = tf.lite.Interpreter(model_content=tflite_quant_model)
ops = sorted({d["op_name"] for d in interpreter._get_ops_details()})
print("Ops in model:", ops)

INFO:tensorflow:Assets written to: C:\Users\gerar\AppData\Local\Temp\tmprxd_j6p6\assets


INFO:tensorflow:Assets written to: C:\Users\gerar\AppData\Local\Temp\tmprxd_j6p6\assets


Saved artifact at 'C:\Users\gerar\AppData\Local\Temp\tmprxd_j6p6'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 99, 43, 1), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(1, 5), dtype=tf.float32, name=None)
Captures:
  2805594987728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2805594989840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2805594990224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2805594989264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2805594991760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2805594992720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2805594993296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2805594994256: TensorSpec(shape=(), dtype=tf.resource, name=None)


c:\Users\gerar\ai-voice-control\model\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Converted fully_trained.model.keras -> converted_model.tflite (84608 bytes)
Ops in model: ['CONV_2D', 'FULLY_CONNECTED', 'MAX_POOL_2D', 'RESHAPE', 'SOFTMAX']


c:\Users\gerar\ai-voice-control\model\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


# To convert to C++
This will run a command line too to convert out tflite model into C code.

In [4]:
from pathlib import Path

input_path = Path("converted_model.tflite")
output_path = Path("model_data.cc")

if not input_path.exists():
    raise FileNotFoundError(f"{input_path} was not found. Run the conversion cell first.")

data = input_path.read_bytes()
var_name = "converted_model_tflite"

lines = []
#lines.append("#include <cstdint>\n")
lines.append(f"extern const unsigned char {var_name}[] = {{\n")
for i in range(0, len(data), 12):
    chunk = data[i:i+12]
    hex_values = ", ".join(f"0x{b:02x}" for b in chunk)
    suffix = "," if i + 12 < len(data) else ""
    lines.append(f"  {hex_values}{suffix}\n")
lines.append("};\n")
lines.append(f"const unsigned int {var_name}_len = {len(data)};\n")

output_path.write_text("".join(lines), encoding="utf-8")
print(f"Wrote {output_path} with {len(data)} bytes")

Wrote model_data.cc with 84608 bytes
